# Luh / KIT RADAR extract notebook

This notebook is the Colab-side extract step for adding the Luh/KIT RADAR dataset to the Dicle battery-transfer pipeline.

It does the heavy raw-data work in Drive/Colab, then downloads only compact reproducibility artifacts:

- `luh_cycles_tidy.csv`
- `luh_cell_metadata.csv`
- `luh_cell_audit.csv`
- `luh_threshold_summary.csv`
- `features_sop12_luh.csv`
- optional `features_sop12_luh_capnorm.csv`
- optional `features_sop12_combined_plus_luh.csv`

The raw 69 GB dataset is not copied into the repo and is not included in the ZIP.

## 0. Configuration

The default `LUH_ROOT` matches the folder used in the earlier exploratory notebook: `/content/drive/MyDrive/RADAR69GB/unzipped/10.35097-1947/data/dataset`.

The notebook uses EOCV2 capacity estimates aligned to `cell_log_age` EFC, then interpolates capacity onto integer equivalent-full-cycle points. This makes the output compatible with the same 34 capacity-only feature definitions used for MATR, HUST, and Sandia.

In [ ]:
# Edit only if your Drive folder or branch name differs.
GITHUB_REPO = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
BRANCH = "main"

LUH_ROOT = "/content/drive/MyDrive/RADAR69GB/unzipped/10.35097-1947/data/dataset"
REPO_DIR = "/content/Graduation-Project-Dicle"

# Match the active thesis feature windows.
N_WINDOWS = [50, 100]
EOL_FRACTION = 0.85
EXTRA_EOL_FRACTIONS_FOR_AUDIT = [0.90, 0.85, 0.80]

# Progress-report-compatible standard cycling subset.
MIN_PARAM_ID = 17
MAX_PARAM_ID = 64
EXCLUDE_RATE_CENTERS = [1.67]
EXCLUDE_RATE_TOL = 0.02
EXCLUDE_0C_1C = False

# EOCV2/log-age column conventions from the prior Luh notebook.
CAPACITY_COL_CANDIDATES = ["cap_aged_est_Ah", "capacity", "cap"]
TIMESTAMP_COL_CANDIDATES = ["timestamp_s", "timestamp", "time_s"]
EFC_COL_CANDIDATES = ["EFC", "efc"]

WRITE_CAPACITY_NORMALIZED_COPY = True
WRITE_COMBINED_PLUS_LUH = True

## 1. Clone the repo and install lightweight dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR)
if REPO_DIR.exists():
    print(f"[clone] repo already exists at {REPO_DIR}; fetching latest {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("[repo]", Path.cwd())

In [ ]:
%pip install -q numpy pandas scipy matplotlib

## 2. Mount Drive and prepare the compact folders

The notebook reads only three extracted folders:

- `cfg_extracted/`
- `cell_eocv2_extracted/`
- `cell_log_age_extracted/`

If an extracted folder is missing, the helper tries to extract the matching archive from `LUH_ROOT`. This avoids touching unrelated raw files.

In [ ]:
from google.colab import drive
from pathlib import Path
import glob
import os
import subprocess

try:
    drive.mount('/content/drive')
except ValueError:
    print('[drive] already mounted')

LUH_ROOT = Path(LUH_ROOT)
if not LUH_ROOT.exists():
    raise FileNotFoundError(f"Luh/RADAR folder not found: {LUH_ROOT}")
print("[luh root]", LUH_ROOT)

subprocess.run(["apt-get", "update", "-y"], check=False)
subprocess.run(["apt-get", "install", "-y", "p7zip-full"], check=False)

CFG_DIR = LUH_ROOT / "cfg_extracted"
EOCV2_DIR = LUH_ROOT / "cell_eocv2_extracted"
LOG_AGE_DIR = LUH_ROOT / "cell_log_age_extracted"

def has_csvs(path: Path, pattern: str) -> bool:
    return path.exists() and any(path.rglob(pattern))

def maybe_extract(name: str, out_dir: Path, archive_patterns: list[str], check_pattern: str) -> None:
    if has_csvs(out_dir, check_pattern):
        print(f"[extract] {name}: using existing {out_dir}")
        return
    archives = []
    for pat in archive_patterns:
        archives.extend(sorted(LUH_ROOT.glob(pat)))
    if not archives:
        print(f"[extract] {name}: no archive found; expected one of {archive_patterns}")
        return
    archive = archives[0]
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {name}: {archive.name} -> {out_dir}")
    subprocess.run(["7z", "x", str(archive), f"-o{out_dir}"], check=True)

maybe_extract("cfg", CFG_DIR, ["cfg.zip", "cfg.7z"], "cell_cfg_*.csv")
maybe_extract("eocv2", EOCV2_DIR, ["cell_eocv2.zip", "cell_eocv2.7z"], "cell_eocv2_*.csv")
maybe_extract("log_age", LOG_AGE_DIR, ["cell_log_age*.zip", "cell_log_age*.7z"], "cell_log_age_*.csv")

cfg_files = sorted(CFG_DIR.rglob("cell_cfg_*.csv"))
eocv2_files = sorted(EOCV2_DIR.rglob("cell_eocv2_*.csv"))
log_age_files = sorted(LOG_AGE_DIR.rglob("cell_log_age_*.csv"))
print(f"[discover] cfg={len(cfg_files)} eocv2={len(eocv2_files)} log_age={len(log_age_files)}")
if not cfg_files or not eocv2_files or not log_age_files:
    raise FileNotFoundError("Missing one or more required extracted folders. Check LUH_ROOT and archive names.")

## 3. Helpers: metadata, file matching, and EFC interpolation

In [ ]:
import importlib.util
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

INTERMEDIATE_DIR = REPO_DIR / "data" / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

feature_module_path = REPO_DIR / "1_features" / "build_features.py"
spec = importlib.util.spec_from_file_location("build_features", feature_module_path)
bf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bf)

KEY_RE = re.compile(r"(P\d{3}_\d+_S\d+_C\d+)")
LOG_RES_RE = re.compile(r"log_age_(\d+)s")

def extract_key(name: str) -> str | None:
    match = KEY_RE.search(str(name))
    return match.group(1) if match else None

def parse_param_id(key: str) -> int:
    return int(key.split("_")[0][1:])

def log_resolution_seconds(path: Path) -> float:
    match = LOG_RES_RE.search(path.name)
    return float(match.group(1)) if match else float("inf")

def choose_col(columns, candidates, contains_fallback=None):
    clean = {str(c).strip(): c for c in columns}
    lowered = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand in clean:
            return clean[cand]
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    if contains_fallback:
        for c in columns:
            lc = str(c).lower()
            if all(token in lc for token in contains_fallback):
                return c
    return None

def read_semicolon(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", low_memory=False, **kwargs)

def rate_is_excluded(rate: float) -> bool:
    if not np.isfinite(rate):
        return False
    return any(abs(rate - center) < EXCLUDE_RATE_TOL for center in EXCLUDE_RATE_CENTERS)

def active_filter_reason(param_id, temp, rate):
    reasons = []
    if not (MIN_PARAM_ID <= param_id <= MAX_PARAM_ID):
        reasons.append("param_id_outside_standard_cycling_range")
    if rate_is_excluded(rate):
        reasons.append("excluded_charge_rate")
    if EXCLUDE_0C_1C and np.isfinite(temp) and temp == 0 and abs(rate - 1.0) < 1e-6:
        reasons.append("excluded_0C_1C")
    return ";".join(reasons) if reasons else "active"

def safe_float(value):
    try:
        out = float(value)
    except Exception:
        return float("nan")
    return out

## 4. Build cell metadata and audit file availability

In [ ]:
eocv2_map = defaultdict(list)
for path in eocv2_files:
    key = extract_key(path.name)
    if key:
        eocv2_map[key].append(path)

log_age_map = defaultdict(list)
for path in log_age_files:
    key = extract_key(path.name)
    if key:
        log_age_map[key].append(path)

records = []
for cfg_path in cfg_files:
    key = extract_key(cfg_path.name)
    if not key:
        continue
    pid = parse_param_id(key)
    rec = {
        "dataset": "luh",
        "cell_id": f"luh_{key}",
        "source_key": key,
        "param_id": pid,
        "cfg_file": str(cfg_path),
        "n_eocv2_files": len(eocv2_map.get(key, [])),
        "n_log_age_files": len(log_age_map.get(key, [])),
        "best_log_age_resolution_s": min([log_resolution_seconds(p) for p in log_age_map.get(key, [])], default=float("nan")),
    }
    try:
        cfg = read_semicolon(cfg_path)
        cfg.columns = cfg.columns.str.strip()
        first = cfg.iloc[0] if len(cfg) else pd.Series(dtype=object)
        temp = safe_float(first.get("age_temp", np.nan))
        rate = safe_float(first.get("age_chg_rate", np.nan))
        rec.update({
            "age_temp": temp,
            "age_chg_rate": rate,
            "age_dischg_rate": safe_float(first.get("age_dischg_rate", np.nan)),
            "age_soc_min": safe_float(first.get("age_soc_min", np.nan)),
            "age_soc_max": safe_float(first.get("age_soc_max", np.nan)),
            "age_dod": safe_float(first.get("age_dod", np.nan)),
            "cfg_columns": json.dumps(list(cfg.columns)),
            "metadata_status": "ok",
        })
    except Exception as exc:
        temp = float("nan")
        rate = float("nan")
        rec.update({
            "age_temp": temp,
            "age_chg_rate": rate,
            "age_dischg_rate": float("nan"),
            "age_soc_min": float("nan"),
            "age_soc_max": float("nan"),
            "age_dod": float("nan"),
            "cfg_columns": "[]",
            "metadata_status": f"error:{type(exc).__name__}",
            "metadata_error": str(exc),
        })
    rec["filter_reason"] = active_filter_reason(pid, rec["age_temp"], rec["age_chg_rate"])
    rec["active_for_features"] = int(rec["filter_reason"] == "active")
    records.append(rec)

metadata = pd.DataFrame(records).sort_values(["param_id", "source_key"])
metadata_path = INTERMEDIATE_DIR / "luh_cell_metadata.csv"
metadata.to_csv(metadata_path, index=False)
print(f"[metadata] {len(metadata)} cfg records -> {metadata_path}")
print(metadata["filter_reason"].value_counts(dropna=False).to_string())
display(metadata.head())

## 5. Parse EOCV2 capacity traces and align them to EFC

For each active cell, this section:

1. selects the finest `cell_log_age_*s_*.csv` file,
2. aligns each EOCV2 capacity estimate to the latest available EFC timestamp,
3. interpolates capacity onto an integer-EFC grid, and
4. computes the same Q0/EOL audit definitions used elsewhere in the repo.

In [ ]:
def load_capacity_trace(cap_path: Path) -> pd.DataFrame:
    df = read_semicolon(cap_path)
    df.columns = df.columns.str.strip()
    cap_col = choose_col(df.columns, CAPACITY_COL_CANDIDATES, contains_fallback=["cap"])
    ts_col = choose_col(df.columns, TIMESTAMP_COL_CANDIDATES, contains_fallback=["timestamp"])
    if cap_col is None or ts_col is None:
        raise ValueError(f"missing capacity/timestamp columns in {cap_path.name}: {list(df.columns)[:12]}")
    out = df[[ts_col, cap_col]].copy()
    out.columns = ["timestamp_s", "capacity_ah"]
    out["timestamp_s"] = pd.to_numeric(out["timestamp_s"], errors="coerce")
    out["capacity_ah"] = pd.to_numeric(out["capacity_ah"], errors="coerce")
    out = out.dropna(subset=["timestamp_s", "capacity_ah"])
    out = out[out["capacity_ah"] > 0]
    return out.sort_values("timestamp_s")

def load_efc_trace(log_path: Path) -> pd.DataFrame:
    # Detect column names from the raw header first. Some RADAR files carry
    # leading/trailing spaces; usecols must receive the raw name, while the
    # selected dataframe is normalized after loading.
    header = read_semicolon(log_path, nrows=0)
    raw_columns = list(header.columns)
    efc_col = choose_col(raw_columns, EFC_COL_CANDIDATES, contains_fallback=["efc"])
    ts_col = choose_col(raw_columns, TIMESTAMP_COL_CANDIDATES, contains_fallback=["timestamp"])
    if efc_col is None or ts_col is None:
        raise ValueError(f"missing EFC/timestamp columns in {log_path.name}: {raw_columns[:12]}")
    df = read_semicolon(log_path, usecols=[ts_col, efc_col])
    df.columns = df.columns.str.strip()
    ts_name = str(ts_col).strip()
    efc_name = str(efc_col).strip()
    out = df[[ts_name, efc_name]].copy()
    out.columns = ["timestamp_s", "EFC"]
    out["timestamp_s"] = pd.to_numeric(out["timestamp_s"], errors="coerce")
    out["EFC"] = pd.to_numeric(out["EFC"], errors="coerce")
    out = out.dropna(subset=["timestamp_s", "EFC"])
    out = out[out["EFC"] >= 0]
    return out.sort_values("timestamp_s")

def capacity_on_integer_efc(cap_df: pd.DataFrame, efc_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    aligned = pd.merge_asof(
        cap_df.sort_values("timestamp_s"),
        efc_df.sort_values("timestamp_s"),
        on="timestamp_s",
        direction="backward",
    ).dropna(subset=["EFC", "capacity_ah"])
    aligned = aligned.sort_values("EFC")
    if aligned.empty or aligned["EFC"].max() < max(N_WINDOWS):
        return aligned, pd.DataFrame(columns=["cycle", "Q_discharge"])

    by_efc = aligned.groupby("EFC", as_index=False)["capacity_ah"].mean().sort_values("EFC")
    efc = by_efc["EFC"].to_numpy(dtype=float)
    cap = by_efc["capacity_ah"].to_numpy(dtype=float)
    finite = np.isfinite(efc) & np.isfinite(cap) & (cap > 0)
    efc = efc[finite]
    cap = cap[finite]
    if len(efc) < 2 or np.nanmax(efc) < max(N_WINDOWS):
        return aligned, pd.DataFrame(columns=["cycle", "Q_discharge"])

    max_cycle = int(math.floor(np.nanmax(efc)))
    grid = np.arange(1, max_cycle + 1, dtype=float)
    q_grid = np.interp(grid, efc, cap)
    tidy = pd.DataFrame({
        "cycle": grid.astype(int),
        "EFC": grid,
        "Q_discharge": q_grid,
    })
    return aligned, tidy

audit_rows = []
tidy_parts = []
active_meta = metadata[metadata["active_for_features"] == 1].copy()

for idx, rec in active_meta.iterrows():
    key = rec["source_key"]
    cell_id = rec["cell_id"]
    row = rec.to_dict()
    row.update({
        "parse_status": "not_started",
        "selected_eocv2_file": "",
        "selected_log_age_file": "",
        "n_eocv2_points": 0,
        "n_aligned_eocv2_points": 0,
        "n_interpolated_cycles": 0,
    })
    try:
        cap_paths = sorted(eocv2_map.get(key, []))
        log_paths = sorted(log_age_map.get(key, []), key=log_resolution_seconds)
        if not cap_paths:
            raise FileNotFoundError("no cell_eocv2 file")
        if not log_paths:
            raise FileNotFoundError("no cell_log_age file")
        cap_path = cap_paths[0]
        log_path = log_paths[0]
        row["selected_eocv2_file"] = str(cap_path)
        row["selected_log_age_file"] = str(log_path)

        cap_df = load_capacity_trace(cap_path)
        efc_df = load_efc_trace(log_path)
        aligned, tidy = capacity_on_integer_efc(cap_df, efc_df)
        row["n_eocv2_points"] = len(cap_df)
        row["n_log_age_points"] = len(efc_df)
        row["n_aligned_eocv2_points"] = len(aligned)
        row["n_interpolated_cycles"] = len(tidy)
        if tidy.empty:
            raise RuntimeError("insufficient aligned EFC/capacity trace for requested N windows")

        qd = tidy["Q_discharge"].to_numpy(dtype=float)
        q0 = bf.compute_q0(qd)
        row["q0"] = q0
        row["last_cycle"] = int(tidy["cycle"].max())
        row["last_qdis"] = float(tidy["Q_discharge"].iloc[-1])
        row["min_qdis"] = float(tidy["Q_discharge"].min())
        row["max_qdis"] = float(tidy["Q_discharge"].max())
        row["raw_eocv2_min_efc"] = float(aligned["EFC"].min()) if len(aligned) else float("nan")
        row["raw_eocv2_max_efc"] = float(aligned["EFC"].max()) if len(aligned) else float("nan")
        for frac in EXTRA_EOL_FRACTIONS_FOR_AUDIT:
            label = f"cycle_life_{str(frac).replace('.', 'p')}"
            row[label] = bf.compute_cycle_life(qd, q0, frac)
        active_label = f"cycle_life_{str(EOL_FRACTION).replace('.', 'p')}"
        row["cycle_life"] = row[active_label]
        row["is_censored"] = int(not np.isfinite(row["cycle_life"]))
        row["parse_status"] = "ok"

        tidy = tidy.assign(
            dataset="luh",
            cell_id=cell_id,
            source_key=key,
            age_temp=rec.get("age_temp", np.nan),
            age_chg_rate=rec.get("age_chg_rate", np.nan),
            param_id=rec.get("param_id", np.nan),
        )
        tidy_parts.append(tidy[["dataset", "cell_id", "source_key", "param_id", "age_temp", "age_chg_rate", "cycle", "EFC", "Q_discharge"]])
    except Exception as exc:
        row["parse_status"] = f"error:{type(exc).__name__}"
        row["error_message"] = str(exc)
    audit_rows.append(row)
    if (len(audit_rows) % 25) == 0:
        print(f"[audit] {len(audit_rows)}/{len(active_meta)} active cells processed")

audit = pd.DataFrame(audit_rows)
cycles_tidy = pd.concat(tidy_parts, ignore_index=True) if tidy_parts else pd.DataFrame()

audit_path = INTERMEDIATE_DIR / "luh_cell_audit.csv"
cycles_path = INTERMEDIATE_DIR / "luh_cycles_tidy.csv"
audit.to_csv(audit_path, index=False)
cycles_tidy.to_csv(cycles_path, index=False)
print(f"[audit] {len(audit)} active rows -> {audit_path}")
print(f"[cycles] {len(cycles_tidy)} interpolated EFC rows -> {cycles_path}")
print(audit["parse_status"].value_counts(dropna=False).to_string())
if not cycles_tidy.empty:
    display(cycles_tidy.head())

## 6. Threshold summary and quick audit plots

In [ ]:
if audit.empty:
    raise RuntimeError("Luh audit is empty. Re-run the metadata and parse cells, then inspect folder paths.")

if "parse_status" not in audit.columns:
    raise RuntimeError("Luh audit has no parse_status column. Re-run the parse cell before building the summary.")

ok = audit[audit["parse_status"] == "ok"].copy()
if ok.empty:
    error_cols = [c for c in ["source_key", "param_id", "age_temp", "age_chg_rate", "parse_status", "error_message", "selected_eocv2_file", "selected_log_age_file"] if c in audit.columns]
    display(audit[error_cols].head(30))
    raise RuntimeError("No Luh cells parsed successfully. Inspect the parse errors above; this is usually a file-layout or column-name mismatch.")

summary_rows = []
for frac in EXTRA_EOL_FRACTIONS_FOR_AUDIT:
    col = f"cycle_life_{str(frac).replace('.', 'p')}"
    if col not in ok.columns:
        ok[col] = np.nan
    reached = ok[col].notna() & np.isfinite(ok[col])
    summary_rows.append({
        "dataset": "luh",
        "eol_fraction": frac,
        "n_cells_ok": len(ok),
        "n_reached_eol": int(reached.sum()),
        "n_censored": int((~reached).sum()),
        "reached_eol_fraction": float(reached.mean()) if len(ok) else float("nan"),
    })
threshold_summary = pd.DataFrame(summary_rows)
threshold_path = INTERMEDIATE_DIR / "luh_threshold_summary.csv"
threshold_summary.to_csv(threshold_path, index=False)
print(threshold_summary.to_string(index=False))

cols = ["param_id", "age_temp", "age_chg_rate", "parse_status", "q0", "cycle_life", "is_censored", "last_cycle", "last_qdis"]
display(audit[[c for c in cols if c in audit.columns]].head(20))
print("\nBy temperature and censoring:")
if "age_temp" in audit.columns and "is_censored" in audit.columns:
    print(ok.groupby(["age_temp", "is_censored"]).size().to_string())


## 7. Build the 34-feature table

This imports the feature functions from `1_features/build_features.py`, so Luh uses the exact same feature definitions as MATR/HUST/Sandia. The only dataset-specific step is the EFC interpolation performed above.

In [ ]:
def qd_by_cell_from_tidy(tidy: pd.DataFrame) -> dict[str, np.ndarray]:
    out = {}
    for cell_id, group in tidy.groupby("cell_id"):
        group = group.sort_values("cycle")
        out[cell_id] = group["Q_discharge"].to_numpy(dtype=float)
    return out

if cycles_tidy.empty:
    raise RuntimeError("No Luh trajectories are available. Fix the audit/parse errors before building features.")

qd_by_cell = qd_by_cell_from_tidy(cycles_tidy)
print(f"[features] cells with interpolated trajectories: {len(qd_by_cell)}")

luh_rows = bf.build_feature_rows(
    qd_by_cell,
    dataset="luh",
    n_windows=tuple(N_WINDOWS),
    eol_fraction=EOL_FRACTION,
    capacity_normalize=False,
)
features_luh = pd.DataFrame(luh_rows)
features_path = INTERMEDIATE_DIR / "features_sop12_luh.csv"
features_luh.to_csv(features_path, index=False)
print(f"[features] wrote {len(features_luh)} rows -> {features_path}")

if WRITE_CAPACITY_NORMALIZED_COPY:
    capnorm_rows = bf.build_feature_rows(
        qd_by_cell,
        dataset="luh",
        n_windows=tuple(N_WINDOWS),
        eol_fraction=EOL_FRACTION,
        capacity_normalize=True,
    )
    features_capnorm = pd.DataFrame(capnorm_rows)
    capnorm_path = INTERMEDIATE_DIR / "features_sop12_luh_capnorm.csv"
    features_capnorm.to_csv(capnorm_path, index=False)
    print(f"[features] wrote {len(features_capnorm)} cap-normalized rows -> {capnorm_path}")

if WRITE_COMBINED_PLUS_LUH:
    combined_base_path = INTERMEDIATE_DIR / "features_sop12_combined.csv"
    if combined_base_path.exists():
        combined_base = pd.read_csv(combined_base_path)
        combined = pd.concat([combined_base, features_luh], ignore_index=True)
        combined_path = INTERMEDIATE_DIR / "features_sop12_combined_plus_luh.csv"
        combined.to_csv(combined_path, index=False)
        print(f"[features] wrote combined table -> {combined_path}")
    else:
        print(f"[features] skipped combined table; missing {combined_base_path}")

if not features_luh.empty:
    display(features_luh.groupby(["dataset", "n_cycles", "is_censored"]).size().rename("n").reset_index())
    display(features_luh.head())

## 8. Make a compact ZIP for local use

This ZIP intentionally contains no raw Luh files.

In [ ]:
import datetime
import zipfile
from google.colab import files

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
zip_path = Path('/content') / f'luh_extract_outputs_{stamp}.zip'

include_files = [
    'data/intermediate/luh_cycles_tidy.csv',
    'data/intermediate/luh_cell_metadata.csv',
    'data/intermediate/luh_cell_audit.csv',
    'data/intermediate/luh_threshold_summary.csv',
    'data/intermediate/features_sop12_luh.csv',
    'data/intermediate/features_sop12_luh_capnorm.csv',
    'data/intermediate/features_sop12_combined_plus_luh.csv',
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel in include_files:
        path = REPO_DIR / rel
        if not path.exists():
            print(f'[skip] {rel} (not produced this run)')
            continue
        zf.write(path, rel)
        print(f'  +{rel}  ({path.stat().st_size / 1024:.1f} KB)')

print(f'\n[zip] wrote {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)')
files.download(str(zip_path))